<a href="https://colab.research.google.com/github/Jackici/Public-repository/blob/main/%E3%80%8CFedMemPath_X_Colab_reproducibility_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FedMemPath-X controlled reproducibility run

**Scope.** This notebook runs a new, deterministic binary linear-update simulation with 20 synthetic clients. It is **not** the archived FedMemPath-X heterogeneous graph experiment and its numerical outputs must be identified as additional controlled evidence. Execute in Colab via Runtime → Change runtime type → GPU; dependencies are preinstalled in standard Colab. Seeds, configuration, the complete attack × non-IID breakdown, per-seed outputs, measured run times, and common-gate governance decisions are exported. The runtime output may differ by GPU and Colab image.

The CIC-MalMem-2022 per-family figure needs the real CSV and genuine sample identifiers; this notebook intentionally does not invent either.

In [ ]:
# Colab: execute in a GPU runtime. This is a NEW controlled linear-update simulation,
# not a replication of the original unarchived graph experiment.
import os, json, time, math
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score
from pathlib import Path
from scipy.stats import median_abs_deviation
OUT=Path('fedmempath_colab_outputs'); OUT.mkdir(exist_ok=True)
SEEDS=[42,52,62,72,82]; ALPHAS=[0.1,0.5,1.0]; ATTACKS=['label_flip','sign_flip','replacement','drift_mimic']; RATIOS=[0.0,0.1,0.2,0.3]
METHODS=['FedAvg','Krum','Median','TrimmedMean','DCTFA','DCTFA_no_predictor']
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('Device:',DEVICE,torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU')
CONFIG=dict(n_clients=20,dim=8,rounds=5,local_n=80,test_n=1200,lr=0.11,drift_scale=.35,
            predictor_train_windows=8,trust_init=.5,rho=.8,gamma=1.5,trim_fraction=.2,
            reject_mad=3.5,min_trust=.05,seed_set=SEEDS,alphas=ALPHAS,attacks=ATTACKS,ratios=RATIOS,
            model='binary logistic regression; synthetic Gaussian evidence vectors, not a graph',
            predictor='ridge fit on trusted enrollment signatures and honest updates; no client labels at inference',
            selection='fixed hyperparameters before held-out attack evaluation',
            asr='among clean-correct malicious test cases, fraction predicted benign after attack')
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2),encoding='utf-8')

def dataset(rng,n,p=0.5,shift=None):
 y=(rng.random(n)<p).astype(np.int64)
 X=rng.normal(0,1,(n,8)).astype('float32')+((2*y-1)[:,None]*np.array([.72,.65,.6,.55,.5,.45,.4,.35],dtype='float32'))
 if shift is not None:X+=shift
 return X,y

def grad(w,X,y):
 x=torch.as_tensor(X,dtype=torch.float32,device=DEVICE); yt=torch.as_tensor(y,dtype=torch.float32,device=DEVICE)
 z=x@w[:8]+w[8];err=torch.sigmoid(z)-yt
 return torch.cat([(x.T@err)/len(X),err.mean().view(1)])

def evaluate(w,X,y):
 with torch.no_grad(): pred=(torch.sigmoid(torch.as_tensor(X,dtype=torch.float32,device=DEVICE)@w[:8]+w[8])>.5).cpu().numpy().astype(int)
 return f1_score(y,pred,zero_division=0),pred

def aggregate(updates,method,pred,signatures,trust):
 u=np.stack(updates); n=len(u); accepted=np.ones(n,dtype=bool);scores=np.zeros(n)
 if method=='FedAvg':return u.mean(0),accepted,scores
 if method=='Median':return np.median(u,axis=0),accepted,scores
 if method=='TrimmedMean':
  k=int(.2*n); return np.sort(u,axis=0)[k:n-k].mean(0),accepted,scores
 if method=='Krum':
  d=((u[:,None,:]-u[None,:,:])**2).sum(-1);f=min(6,n//3); neighbor=np.sort(d,axis=1)[:,1:n-f-1].sum(1)
  idx=int(np.argmin(neighbor));accepted[:]=False;accepted[idx]=True;scores=neighbor
  return u[idx],accepted,scores
 residual=u-(signatures@pred if method=='DCTFA' else 0)
 center=np.median(residual,axis=0);scores=np.linalg.norm(residual-center,axis=1)
 med=np.median(scores);mad=median_abs_deviation(scores,scale='normal')+1e-8
 accepted=scores<=med+CONFIG['reject_mad']*mad
 if not accepted.any():accepted[np.argmin(scores)]=True
 trust[:]=CONFIG['rho']*trust+(1-CONFIG['rho'])*np.exp(-np.clip(scores,0,10))
 weight=trust*np.exp(-CONFIG['gamma']*np.clip(scores,0,10))*accepted
 return np.average(u,axis=0,weights=weight+1e-12),accepted,scores

def one(seed,alpha,attack,ratio,method):
 rng=np.random.default_rng(seed+int(alpha*1000)+ATTACKS.index(attack)*10000)
 Xte,yte=dataset(np.random.default_rng(seed+79001),CONFIG['test_n'])
 # identical clean baseline and attack schedule for all methods in this cell
 client_p=rng.dirichlet([alpha,alpha],size=20)[:,1]
 shifts=rng.normal(0,CONFIG['drift_scale'],(20,8)).astype('float32')
 honest_updates=[];sig=[]
 w0=torch.zeros(9,device=DEVICE, dtype=torch.float32)
 for j in range(20):
  for h in range(CONFIG['predictor_train_windows']):
   XX,yy=dataset(rng,80,float(client_p[j]),shifts[j]*(h+1)/8)
   ww=w0-grad(w0,XX,yy)*CONFIG['lr']; honest_updates.append(ww.cpu().numpy());sig.append(np.r_[XX.mean(0),1.])
 # predictor fitted solely on the pre-attack trusted enrollment period
 A=np.asarray(sig);B=np.asarray(honest_updates); predictor=np.linalg.solve(A.T@A+0.1*np.eye(9),A.T@B)
 chosen=set(rng.choice(20,int(round(20*ratio)),replace=False).tolist())
 w=torch.zeros(9,device=DEVICE, dtype=torch.float32); clean_w=w.clone();trust=np.full(20,CONFIG['trust_init'])
 for r in range(CONFIG['rounds']):
  batches=[];deltas=[];ss=[]
  for j in range(20):
   XX,yy=dataset(rng,80,float(client_p[j]),shifts[j]*(r+1)/CONFIG['rounds'])
   batches.append((XX,yy));ss.append(np.r_[XX.mean(0),1.])
   clean=-(CONFIG['lr']*grad(w,XX,yy)).cpu().numpy()
   if j in chosen:
    if attack=='label_flip':u=-(CONFIG['lr']*grad(w,XX,1-yy)).cpu().numpy()
    elif attack=='sign_flip':u=-clean
    elif attack=='replacement':u=6*clean+np.r_[[-.7]*8,2.].astype('float32')
    else:u=clean+np.r_[[-.09]*8,.25].astype('float32')
   else:u=clean
   deltas.append(u)
  base=np.mean([-(CONFIG['lr']*grad(clean_w,XX,yy)).cpu().numpy() for XX,yy in batches],axis=0)
  clean_w+=torch.as_tensor(base,device=DEVICE)
  agg,accepted,scores=aggregate(deltas,method,predictor,np.array(ss),trust)
  w+=torch.as_tensor(agg,device=DEVICE)
 f1,p=evaluate(w,Xte,yte);_,clean_pred=evaluate(clean_w,Xte,yte)
 target=(yte==1)&(clean_pred==1);asr=float(np.mean(p[target]==0)) if target.any() else np.nan
 ids=np.array([j in chosen for j in range(20)]); auroc=float(roc_auc_score(ids,scores)) if ids.any() and (~ids).any() and len(np.unique(scores))>1 else np.nan
 honest=np.array([not x for x in ids]); hdfr=float(np.mean(~accepted[honest]))
 return dict(seed=seed,alpha=alpha,attack=attack,malicious_ratio=ratio,method=method,attacked_f1=f1,
             asr=asr,malicious_auroc=auroc,honest_drift_false_rejection=hdfr,
             rejected=int((~accepted).sum()),n_attack_targets=int(target.sum()))

rows=[];start=time.perf_counter()
for seed in SEEDS:
 for alpha in ALPHAS:
  for attack in ATTACKS:
   for ratio in RATIOS:
    for method in METHODS:
     t=time.perf_counter();x=one(seed,alpha,attack,ratio,method)
     if DEVICE.type=='cuda':torch.cuda.synchronize()
     x['wall_seconds']=time.perf_counter()-t;rows.append(x)
  print('Completed',seed,alpha,'elapsed',round(time.perf_counter()-start,1),'s')
raw=pd.DataFrame(rows); raw.to_csv(OUT/'federated_per_seed.csv',index=False)
summary=raw.groupby(['method','attack','alpha','malicious_ratio'],dropna=False).agg(
 attacked_f1_mean=('attacked_f1','mean'),attacked_f1_sd=('attacked_f1','std'),
 asr_mean=('asr','mean'),malicious_auroc_mean=('malicious_auroc','mean'),
 hdfr_mean=('honest_drift_false_rejection','mean'),wall_seconds_mean=('wall_seconds','mean')).reset_index()
summary.to_csv(OUT/'federated_by_attack_alpha.csv',index=False)
assert raw.loc[raw.malicious_ratio==0,'malicious_auroc'].isna().all()
print('Wrote',len(raw),'seed-level rows and',len(summary),'stratified rows.')

# Identical final policy admissibility is enforced for every governance arm.
GOV_METHODS=['Static','ModelOnly','ZTGov','NoUncertainty','NoCriticality']
def governance(seed):
 rng=np.random.default_rng(seed+35000);n=2000
 truth=rng.random(n)<.28; risk=np.clip(.2+.6*truth+rng.normal(0,.18,n),0,1)
 uncertainty=rng.uniform(0,1,n);critical=rng.integers(0,3,n)
 rows=[]
 for method in GOV_METHODS:
  if method=='Static': proposed=np.where(risk>.7,2,0)
  elif method=='ModelOnly': proposed=np.where(risk>.55,2,0)
  else:
   score=risk+(0 if method=='NoCriticality' else .07*critical)
   review=(risk>.48)&(risk<.73)&(uncertainty>.45) if method!='NoUncertainty' else np.zeros(n,bool)
   proposed=np.where(review,3,np.where(score>.69,2,np.where(score>.45,1,0)))
  # 0 allow, 1 reversible step-up, 2 deny, 3 analyst review.
  # Hard policy forbids automated denial of high-criticality resources.
  forbidden=(proposed==2)&(critical==2)
  executed=np.where(forbidden,3,proposed)
  bad=(executed==2)&(critical==2)
  unsafe=(executed==2)&(~truth)
  rows.append(dict(seed=seed,method=method,n=n,proposed_violation_rate=float(forbidden.mean()),
                   executed_policy_violation_rate=float(bad.mean()),unsafe_action_rate=float(unsafe.mean()),
                   review_rate=float((executed==3).mean())))
 return rows
G=pd.DataFrame([x for seed in SEEDS for x in governance(seed)])
assert (G.executed_policy_violation_rate==0).all()
G.to_csv(OUT/'governance_per_seed.csv',index=False)
G.groupby('method').agg(['mean','std']).to_csv(OUT/'governance_summary.csv')

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':160,'font.size':9})
s=raw.groupby(['method','malicious_ratio'],as_index=False).agg(mean=('attacked_f1','mean'),sd=('attacked_f1','std'))
fig,ax=plt.subplots(figsize=(6,3.4))
for m,g in s.groupby('method'):
 ax.errorbar(g.malicious_ratio*100,g['mean'],yerr=g.sd,label=m,marker='o',capsize=2)
ax.set(xlabel='Malicious clients (%)',ylabel='Synthetic test macro-F1');ax.legend(fontsize=6);ax.grid(alpha=.25);fig.tight_layout();fig.savefig(OUT/'Figure_R2_new_simulation.png');plt.close(fig)
g=G.groupby('method').unsafe_action_rate.agg(['mean','std']).reindex(GOV_METHODS)
fig,ax=plt.subplots(figsize=(6,3.2));ax.bar(g.index,g['mean'],yerr=g['std'],capsize=2);ax.tick_params(axis='x',rotation=15);ax.set(ylabel='Unsafe action rate under common hard gate');fig.tight_layout();fig.savefig(OUT/'Figure_R3_new_simulation.png');plt.close(fig)
print('Governance records:',len(G),'executed violations:',G.executed_policy_violation_rate.sum())
print('Outputs:',[x.name for x in OUT.iterdir()])

Device: cuda NVIDIA A100-SXM4-40GB
Completed 42 0.1 elapsed 17.9 s
Completed 42 0.5 elapsed 35.4 s
Completed 42 1.0 elapsed 52.9 s
Completed 52 0.1 elapsed 70.5 s
Completed 52 0.5 elapsed 88.0 s
Completed 52 1.0 elapsed 105.5 s
Completed 62 0.1 elapsed 123.1 s
Completed 62 0.5 elapsed 140.6 s
Completed 62 1.0 elapsed 158.2 s
Completed 72 0.1 elapsed 175.9 s
Completed 72 0.5 elapsed 193.6 s
Completed 72 1.0 elapsed 211.2 s
Completed 82 0.1 elapsed 228.9 s
Completed 82 0.5 elapsed 246.4 s
Completed 82 1.0 elapsed 264.0 s
Wrote 1440 seed-level rows and 288 stratified rows.
Governance records: 25 executed violations: 0.0
Outputs: ['governance_summary.csv', 'config.json', 'Figure_R2_new_simulation.png', 'Figure_R3_new_simulation.png', 'federated_by_attack_alpha.csv', 'federated_per_seed.csv', 'governance_per_seed.csv']


## Import newly generated results into the manuscript\nRun this cell after the experiment to append correctly labeled tables and figures.\n

In [ ]:
# Add the newly run tables and figures to the revised manuscript as a separately labeled appendix.
# Upload FedMemPath_X_revised_review.docx when prompted.
from google.colab import files
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','-q','install','python-docx'])
from docx import Document
from docx.shared import Inches
uploaded=files.upload()
name=next((k for k in uploaded if k.lower().endswith('.docx')),None)
if name is None: raise ValueError('Upload the revised manuscript DOCX')
doc=Document(name)
doc.add_heading('Appendix B New independently reproducible synthetic controls',level=1)
doc.add_paragraph('The following outputs are from the accompanying binary linear-update simulation, not from the original heterogeneous graph or CIC-MalMem-2022 experiments. Results should not be compared as a direct replication of Tables R1–R4. Each row is computed from the archived per-seed CSV; malicious-client AUROC is undefined when there are no malicious clients.')
doc.add_heading('Attack and non-IID breakdown',level=2)
doc.add_paragraph('The complete attack × Dirichlet α × malicious-ratio × method grid is provided in federated_by_attack_alpha.csv. The table below gives an illustrative compact 30% attack breakdown; consult the CSV for all settings.')
sub=summary[(summary.malicious_ratio==.3)&(summary.alpha==.5)]
t=doc.add_table(rows=1,cols=5);t.style='Table Grid'
for c,s in zip(t.rows[0].cells,['Method','Attack','Macro-F1 mean ± SD','ASR','HDFR']):c.text=s
for _,r in sub.iterrows():
 for c,s in zip(t.add_row().cells,[r['method'],r['attack'],f"{r['attacked_f1_mean']:.3f} ± {r['attacked_f1_sd']:.3f}",f"{r['asr_mean']:.3f}",f"{r['hdfr_mean']:.3f}"]):c.text=s
doc.add_paragraph('Figure B1. New binary linear-update simulation. Five-seed dispersion is computed from all attack and non-IID cells; refer to the stratified CSV for conditional results.')
doc.add_picture(str(OUT/'Figure_R2_new_simulation.png'),width=Inches(5.8))
doc.add_paragraph('Figure B2. Common-hard-gate governance simulation. Executed policy violations equal zero in every method; proposed-action violations are reported separately in governance_per_seed.csv.')
doc.add_picture(str(OUT/'Figure_R3_new_simulation.png'),width=Inches(5.8))
doc.add_paragraph('Reproduction: configuration and seeds are recorded in config.json. Per-seed files: federated_per_seed.csv and governance_per_seed.csv. Timings are measured locally in this run and are not the assumed multipliers in legacy Table R3.')
doc.save('FedMemPath_X_with_new_simulation_appendix.docx')
import shutil
shutil.make_archive('FedMemPath_X_run_outputs','zip',OUT)
files.download('FedMemPath_X_with_new_simulation_appendix.docx')
files.download('FedMemPath_X_run_outputs.zip')

## Optional five-seed family figure\nUpload actual per-family results exported by the real-data pilot.\n

In [ ]:
# Optional real-data five-seed family chart: upload rows with seed,family,model,f1.
# The notebook refuses an incomplete seed × family × model matrix.
from google.colab import files
import io
up=files.upload()
if up:
 df=pd.read_csv(io.BytesIO(next(iter(up.values()))))
 required={'seed','family','model','f1'}
 assert required<=set(df.columns),f'Missing columns: {required-set(df.columns)}'
 assert set(df.seed.unique())==set(SEEDS),'Need exactly all five specified seeds'
 assert (df.groupby(['family','model']).seed.nunique()==5).all(),'Each family/model needs all five seeds'
 agg=df.groupby(['family','model']).f1.agg(['mean','std']).reset_index()
 agg.to_csv(OUT/'family_f1_five_seeds.csv',index=False)
 fig,ax=plt.subplots(figsize=(9,4));
 for m,g in agg.groupby('model'):
  ax.errorbar(g.family,g['mean'],yerr=g['std'],fmt='o',label=m,capsize=2)
 ax.tick_params(axis='x',rotation=75);ax.set(ylabel='Family F1, mean ± SD');ax.legend();fig.tight_layout();fig.savefig(OUT/'Figure_6_five_seed_family.png');plt.show()